# STEP 4-A — 베이스라인 학습

## 왜 굳이 "약한" 모델부터?

최신 모델을 바로 돌리고 싶겠지만, 먼저 **ResNet50 같은 평범한 모델**을 돌려야 합니다.
이유는 두 가지:

1. **파이프라인 검증** — 랜덤(1/6 ≈ 16.7%)보다 못 나오면 모델 문제가 아니라
   라벨/크롭/분할 어딘가가 깨진 겁니다. 무거운 모델로 3시간 태우기 전에 알아야죠.
2. **비교 기준** — "EVA-02가 macro-F1 0.78" 은 그 자체로는 아무 의미가 없습니다.
   "ResNet50이 0.71인데 EVA-02가 0.78" 이어야 판단이 됩니다.

## 이 노트북에서 하는 일

1. ResNet50 으로 파이프라인이 살아있는지 확인
2. **크롭 방식 3종 비교** → 이후 실험에 쓸 크롭 확정
3. 학습 곡선 읽는 법 익히기

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 clone → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"          # 작업 브랜치를 쓰려면 여기만 바꾸세요
DIR    = "deeplearning_test"

if os.path.basename(os.getcwd()) != DIR and not os.path.exists("src"):
    if os.path.exists(DIR):
        subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO], check=True)
    os.chdir(DIR)
sys.path.insert(0, os.getcwd())
print("작업 디렉터리:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
if not os.path.exists("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

In [ ]:
from src import labels, split, data, models, train, evaluate

cfg = CFG(
    model_name="resnet50",
    img_size=224,
    epochs=10,
    exp_name="baseline_resnet50_m1.5",
)
print("배치 크기 (자동):", cfg.resolved_batch_size())

## 1. 데이터 로드

In [ ]:
df = labels.load(env.work_root()/"manifests"/"manifest_m1.5.parquet")
tr, va = split.get_fold(df, cfg.use_fold)
print(f"train {len(tr):,} / val {len(va):,}")

## 2. 모델 만들기

`pretrained=True` 가 핵심입니다. ImageNet 으로 미리 학습된 가중치를 가져와서
우리 데이터로 **미세조정(fine-tuning)** 합니다.

밑바닥부터 학습하려면 수백만 장이 필요하지만, 전이학습을 쓰면 수만 장으로도 됩니다.
사전학습 모델이 이미 "가장자리 → 질감 → 패턴" 을 볼 줄 알기 때문입니다.

📖 [`docs/basics/04_전이학습과_파인튜닝.md`](../docs/basics/04_전이학습과_파인튜닝.md)

In [ ]:
models.available()      # 이 환경의 timm 에서 실제로 쓸 수 있는 모델 확인

In [ ]:
model = models.build("resnet50", n_classes=len(CLASSES),
                     pretrained=True, drop_rate=cfg.drop_rate)

## 3. DataLoader

증강(augmentation)이 여기 들어갑니다.

> ⚠️ 색상 증강을 약하게 잡아둔 이유: **피부 병변은 색이 곧 라벨**입니다.
> 일반 이미지 분류에서 쓰는 `ColorJitter(0.4)` 를 그대로 쓰면
> A3(과다색소침착)의 어두운 색을 밝게 만들어 A1처럼 보이게 합니다.
> 증강이 라벨을 파괴하는 거죠.

In [ ]:
dl_tr, dl_va, ds_tr, ds_va = data.build_loaders(tr, va, cfg, model=model)

In [ ]:
# 증강이 실제로 뭘 하는지 눈으로 보기
import matplotlib.pyplot as plt, torch
from src.data import IMAGENET_MEAN, IMAGENET_STD

x, y = next(iter(dl_tr))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))
for ax, i in zip(axes.flat, range(min(8, len(x)))):
    ax.imshow((x[i]*std+mean).clamp(0,1).permute(1,2,0)); ax.axis("off")
    ax.set_title(f"{CLASSES[y[i]]} {CLASS_KO[CLASSES[y[i]]][:8]}", fontsize=8)
plt.suptitle("증강 후 실제로 모델이 보는 이미지"); plt.tight_layout(); plt.show()
print("💡 병변이 잘려 나가거나 색이 심하게 변했다면 증강이 너무 센 겁니다.")

## 4. 학습

📖 학습 루프 내부가 궁금하면 [`docs/basics/05_학습루프_옵티마이저_스케줄러.md`](../docs/basics/05_학습루프_옵티마이저_스케줄러.md)

In [ ]:
res = train.fit(model, dl_tr, dl_va, cfg, ds_train=ds_tr)

In [ ]:
res.plot()

### 학습 곡선 읽는 법

| 증상 | 의미 | 대응 |
|---|---|---|
| train↓ val↓ 둘 다 계속 하락 | 정상, 더 학습 가능 | epochs 늘리기 |
| train↓ **val↑** | 과적합 시작 | 조기종료 지점, 증강↑ / drop_rate↑ |
| 둘 다 안 내려감 | 학습이 안 됨 | lr 조정, 데이터/라벨 확인 |
| val 이 심하게 출렁임 | 배치가 작거나 lr 이 큼 | batch↑ 또는 lr↓ |

## 5. 평가

In [ ]:
_, logits, ys = train.evaluate_loader(model, dl_va, None, "cuda",
                                      len(CLASSES), tta_hflip=cfg.tta_hflip)
rep = evaluate.full_report(logits, ys, CLASSES)

In [ ]:
rep.plot_confusion()

In [ ]:
rep.plot_per_class()

### 🚦 게이트 체크

**macro-F1 이 0.25 미만이면 여기서 멈추고 데이터를 다시 보세요.**
랜덤이 0.167 인데 그것보다 조금 나은 수준이면 파이프라인 어딘가가 깨진 겁니다.

흔한 원인: 라벨 매칭 오류, 크롭 좌표 오류, 클래스 매핑 뒤바뀜.

In [ ]:
assert rep.metrics["macro_f1"] > 0.25, (
    "베이스라인이 랜덤 수준입니다. 모델을 바꾸지 말고 STEP 2~3을 다시 확인하세요."
)
print("✅ 파이프라인 정상 — 최신 모델 실험으로 넘어가도 됩니다.")

## 6. 크롭 방식 비교 실험 ★

이제 **어떤 크롭이 제일 좋은지** 정합니다. 가벼운 모델로 짧게 돌려 비교합니다.
(이 결과가 이후 모든 실험의 전제가 되므로 먼저 정해야 합니다)

In [ ]:
crop_results = {}
quick = CFG(model_name="tf_efficientnetv2_s.in21k_ft_in1k",
            img_size=224, epochs=6, monitor="macro_f1")

for tag in ["m1.5", "m2.5", "full"]:
    print(f"\n{'='*60}\n  크롭: {tag}\n{'='*60}")
    d = labels.load(env.work_root()/"manifests"/f"manifest_{tag}.parquet")
    t, v = split.get_fold(d, 0)
    m = models.build("effnetv2_s", len(CLASSES), pretrained=True, verbose=False)
    ltr, lva, dtr, _ = data.build_loaders(t, v, quick, model=m)
    r = train.fit(m, ltr, lva, CFG(**{**quick.to_dict(), "exp_name": f"crop_{tag}"}),
                  ds_train=dtr, verbose=True)
    _, lg, yy = train.evaluate_loader(m, lva, None, "cuda", len(CLASSES))
    crop_results[tag] = evaluate.full_report(lg, yy, CLASSES, show=False)
    del m; import torch; torch.cuda.empty_cache()

evaluate.compare_models(crop_results)

In [ ]:
BEST_CROP = max(crop_results, key=lambda k: crop_results[k].metrics["macro_f1"])
print(f"\n✅ 최적 크롭: {BEST_CROP}")
print("   → 04 노트북에서 이 크롭으로 최신 모델들을 비교합니다.")
(env.work_root()/"best_crop.txt").write_text(BEST_CROP)

---
## ✅ 다음 단계

`04_학습_최신모델_비교.ipynb`

📖 함께 읽기: [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md)